In [ ]:
SYSTEM_PROMPT = """You are a helpful assistant that answers questions about Pokemon.
You are friendly, precise and to the point. The only source of truth for you is the
answers you get back from your tools, and you always admit not knowing something
if it is not reflected in what you obtained from your tools, explaining to the
user what the problem was. Offer no extra information.
"""

In [ ]:
import os
import re
from typing_extensions import TypedDict, Annotated
from typing import Literal
import operator

from dotenv import load_dotenv

from sqlalchemy import create_engine, text
from psycopg_pool import ConnectionPool

import polars as pl

from langchain.tools import tool
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_core.messages import SystemMessage, HumanMessage, AnyMessage, ToolMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import MessagesState
from langgraph.prebuilt import ToolNode, tools_condition

from langfuse import get_client
from langfuse.langchain import CallbackHandler

In [ ]:
load_dotenv()

In [ ]:
langfuse = get_client()

if langfuse.auth_check() == True:
    print("Successful auth!")
    langfuse_handler = CallbackHandler()
else:
    raise PermissionError("Failed LangFuse authentication!")

In [ ]:
pool = ConnectionPool(
    f"host=10.8.0.1 port=5433 dbname=pokeapi user=ash password={os.getenv('POSTGRES_PASSWORD')}",
    open = True
)

In [ ]:
from src.tools import make_tools

llm = HuggingFaceEndpoint(
    model="Qwen/Qwen3-4B-Instruct-2507",
    task="text-generation",
    huggingfacehub_api_token=os.environ["HUGGINGFACEHUB_API_TOKEN"],
)

model = ChatHuggingFace(llm=llm)

tools = make_tools(pool)
tools_by_name = {tool.name: tool for tool in tools}

model_with_tools = model.bind_tools(tools)

class MessagesState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]

In [ ]:
# LLM node
def llm_call(state: dict):
    """LLM decides whether to call a tool or not"""
    return {
        "messages": [
            model_with_tools.invoke(
                [
                    SystemMessage(
                        content=SYSTEM_PROMPT
                    )
                ] + state["messages"]
            )
        ],
    }

In [ ]:
# Tool node
def tool_node(state: dict):
    """Performs the tool call"""

    result = []
    for tool_call in state["messages"][-1].tool_calls:
        tool = tools_by_name[tool_call["name"]]
        observation = tool.invoke(tool_call["args"])
        result.append(ToolMessage(content=observation, tool_call_id=tool_call["id"]))
    return {"messages": result}

In [ ]:
# END logic

def should_continue(state: MessagesState) -> Literal["tool_node", END]:
    """Decide if we should continue the loop or stop based upon whether the LLM made a tool call"""

    messages = state["messages"]
    last_message = messages[-1]

    # If the LLM makes a tool call, then perform an action
    if last_message.tool_calls:
        return "tool_node"

    # Otherwise, we stop (reply to the user)
    return END

In [ ]:
# Build workflow
agent_builder = StateGraph(MessagesState)

# Add nodes
agent_builder.add_node("llm_call", llm_call)
agent_builder.add_node("tool_node", tool_node)

# Add edges to connect nodes
agent_builder.add_edge(START, "llm_call")
agent_builder.add_conditional_edges(
    "llm_call",
    should_continue,
    ["tool_node", END]
)
agent_builder.add_edge("tool_node", "llm_call")

# Compile the agent
agent = agent_builder.compile()

# Show the agent
from IPython.display import Image, display
display(Image(agent.get_graph(xray=True).draw_mermaid_png()))

In [ ]:
# Invoke
messages = [HumanMessage(content="I want to find the movements learnt by Pikachu.")]
messages = agent.invoke(
    {"messages": messages},
    config={
        "callbacks": [langfuse_handler],
    },
)
for m in messages["messages"]:
    m.pretty_print()